In [ ]:
import csv
import os
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import Subset,ConcatDataset, Dataset, DataLoader, random_split
from torchvision.io.video import read_video
from torchvision.models.video import r2plus1d_18, R2Plus1D_18_Weights
from pathlib import Path

## Device configuration

In [ ]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Using the device: {device}")

## Dataset definition

In [ ]:
class CutDataset(Dataset):
    def __init__(self, good_dir=None, bad_dir=None, no_cuts_dir=None, transform=None):
        self.samples = []
        self.transform = transform
        valid_ext = {'.mp4', '.avi', '.mov'}
        
        # Helper per caricare i file gestendo l'eccezione della directory mancante
        def add_from_dir(directory, label):
            if directory is None:
                return
            d_path = Path(directory)
            if not d_path.exists():
                print(f"Warning: Directory {directory} non trovata.")
                return
            
            for path in d_path.iterdir():
                if path.is_file() and path.suffix.lower() in valid_ext:
                    self.samples.append((str(path), label))

        # Labels: Bad = 0, Good = 1, No_Cuts = 2
        add_from_dir(bad_dir, 0)
        add_from_dir(good_dir, 1)
        add_from_dir(no_cuts_dir, 2)
            
    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        video_path, label = self.samples[idx]
        
        try:
            # Lettura video
            video, _, _ = read_video(video_path, output_format="TCHW", pts_unit="sec")
            
            # Se il video è vuoto o non caricato correttamente
            if video.size(0) == 0:
                raise ValueError(f"Video vuoto: {video_path}")

            target_frames = 16
            current_frames = video.shape[0]
            
            # Gestione frame (Clip o Padding)
            if current_frames > target_frames:
                video = video[:target_frames] 
            elif current_frames < target_frames:
                last_frame = video[-1:]
                diff = target_frames - current_frames
                padding = last_frame.repeat(diff, 1, 1, 1)
                video = torch.cat((video, padding), dim=0)

            # Normalizzazione
            video = video.to(torch.float32) / 255.0
                
            if self.transform:
                video = self.transform(video)
                
            return video, torch.tensor(label, dtype=torch.long)

        except Exception as e:
            print(f"Errore caricamento video {video_path}: {e}")
            # Se un video fallisce, restituiamo un altro campione a caso per non fermare il batch
            return self.__getitem__(random.randint(0, len(self) - 1))

## Model preparation

In [ ]:
weights = R2Plus1D_18_Weights.DEFAULT
model = r2plus1d_18(weights=weights)

num_classes = 3
model.fc = nn.Linear(model.fc.in_features, num_classes)

'''
#dropout
in_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(p=0.5),
    nn.Linear(in_features, num_classes)
)
'''

model = model.to(device)
preprocess = weights.transforms()

## Data loading

We choice 80% of videos for training and 20% to testing

In [ ]:

full_dataset = CutDataset(good_dir="Good_cuts", bad_dir="Bad_cuts", no_cuts_dir="No_cuts", transform=preprocess)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)


## Training setup

In [ ]:
optimizer = optim.AdamW(model.parameters(), lr=0.0001, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=4)
weightss = torch.tensor([1.0, 1.0, 1.0]).to(device)
criterion = nn.CrossEntropyLoss(weight=weightss)

## Training phase

In [ ]:
num_epochs = 200
best_val_loss = float('inf')
log_filename = "training_logs.csv"

with open(log_filename, mode='w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['Epoch', 'Train_Loss', 'Train_Acc', 'Val_Loss', 'Val_Acc', 'Saved_Best', 'Current_LR'])

print(f"Starting training... Logs will be saved in {log_filename}")

for epoch in range(num_epochs):
    model.train() 
    running_loss = 0.0
    correct = 0
    total = 0
    
    for i, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    train_loss = running_loss / len(train_loader)
    train_acc = 100 * correct / total

    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
            
    val_loss = val_loss / len(val_loader)
    val_acc = 100 * val_correct / val_total

    scheduler.step(val_loss)   
    current_lr = optimizer.param_groups[0]['lr']
    # --- LOGICA DI SALVATAGGIO MODELLO ---
    saved_best = False
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), f"best_model_cuts{epoch+1}.pth")
        saved_best = True
    
    # --- SCRITTURA LOG SU CSV ---
    # Appendiamo i dati dell'epoca corrente al file
    with open(log_filename, mode='a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([epoch+1, f"{train_loss:.4f}", f"{train_acc:.2f}", f"{val_loss:.4f}", f"{val_acc:.2f}", saved_best, f"{current_lr:.6f}"])

    # Stampiamo comunque a video (se mai dovessi riattaccarti alla console)
    print(f"Epoch [{epoch+1}/{num_epochs}] Logged to CSV.")

torch.save(model.state_dict(), "model_cuts.pth")
print("Training completed!")